# Descarga y preparación de datos climáticos

## Proyecto Agronomía

Esta libreta documenta la obtención de los datos climáticos históricos
utilizados para estudiar los factores asociados con el éxito de la
producción agrícola en los municipios de México entre **2003 y 2025**.

Los datos climáticos se obtienen mediante **Open-Meteo Historical Weather API**
para los municipios que presentan actividad agrícola dentro del periodo
de estudio.

Para realizar las consultas climáticas se utilizan coordenadas
representativas obtenidas previamente a partir de las geometrías
municipales de **INEGI**.

La lógica de adquisición, transformación y almacenamiento se encuentra
implementada en el paquete `proyecto_agronomia`. Esta libreta se utiliza
para ejecutar el flujo de trabajo, inspeccionar sus resultados y
documentar el proceso de forma reproducible.

### Variables climáticas

Para cada municipio se obtienen las siguientes variables:

- Temperatura media.
- Temperatura máxima.
- Temperatura mínima.
- Precipitación acumulada.
- Días con lluvia.
- Horas de precipitación.
- Evapotranspiración de referencia FAO (ET₀).

Los datos diarios son posteriormente agregados a escala anual.

### Unidad de análisis

Cada observación climática queda identificada mediante:

`(Anio, Idestado, Idmunicipio)`

Por lo tanto, el resultado contiene una observación por municipio y año.

## Flujo de trabajo

La construcción del conjunto de datos climáticos se realiza mediante
las siguientes etapas:

1. Identificar los municipios que presentan actividad agrícola entre
   2003 y 2025.
2. Cargar las coordenadas representativas obtenidas a partir de INEGI.
3. Asociar cada municipio agrícola con su latitud y longitud.
4. Consultar los datos climáticos históricos mediante Open-Meteo.
5. Resumir las observaciones diarias a escala anual.
6. Guardar los resultados por entidad federativa.
7. Verificar la cobertura temporal y municipal obtenida.

La descarga climática utiliza archivos de progreso por estado. Esto
permite reanudar el proceso sin volver a consultar los municipios que
ya cuentan con información completa.

In [1]:
import pandas as pd

from proyecto_agronomia.config import (
    ANIO_INICIAL,
    ANIO_FINAL,
    INEGI_INTERIM_DIR,
)

from proyecto_agronomia.dataset import (
    descargar_clima_todos_estados,
)

from proyecto_agronomia.features import (
    obtener_municipios_agricolas,
    preparar_municipios_clima,
)

2026-10-02 17:03:25.941 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


## 1. Municipios con actividad agrícola

La descarga climática se limita a los municipios que presentan registros
de actividad agrícola durante el periodo **2003–2025**.

En lugar de consultar información climática para todos los municipios
disponibles en INEGI, primero se obtiene el conjunto de municipios que
aparecen en los datos agrícolas del proyecto.

Los municipios se identifican mediante la combinación de:

- `Idestado`
- `Idmunicipio`

Esta combinación funcionará posteriormente como parte de la llave para
integrar los datos agrícolas, geográficos y climáticos.

In [2]:
municipios_agricolas = obtener_municipios_agricolas()

municipios_agricolas.head()

2026-10-02 17:03:31.743 | INFO     | proyecto_agronomia.features:obtener_municipios_agricolas:402 - Municipios con actividad agrícola: 2,452


,Idestado,Idmunicipio,Nommunicipio
0,1,1,Aguascalientes
1,1,5,Jesús María
2,1,10,El Llano
3,1,11,San Francisco de Los Romo
4,1,3,Calvillo


In [3]:
print(
    f"Periodo de estudio: "
    f"{ANIO_INICIAL}-{ANIO_FINAL}"
)

print(
    f"Municipios con actividad agrícola: "
    f"{len(municipios_agricolas):,}"
)

print(
    f"Entidades federativas representadas: "
    f"{municipios_agricolas['Idestado'].nunique()}"
)

municipios_agricolas.info()

Periodo de estudio: 2003-2025
Municipios con actividad agrícola: 2,452
Entidades federativas representadas: 32
<class 'pandas.DataFrame'>
RangeIndex: 2452 entries, 0 to 2451
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Idestado      2452 non-null   int64
 1   Idmunicipio   2452 non-null   int64
 2   Nommunicipio  2452 non-null   str  
dtypes: int64(2), str(1)
memory usage: 57.6 KB


In [4]:
duplicados = municipios_agricolas.duplicated(
    subset=["Idestado", "Idmunicipio"]
).sum()

print(
    f"Claves municipales duplicadas: {duplicados:,}"
)

Claves municipales duplicadas: 0


### Resultado

Se identificó el conjunto de municipios con actividad agrícola que será
utilizado para la consulta climática.

Cada municipio aparece una sola vez mediante la llave
`(Idestado, Idmunicipio)`.

En la siguiente etapa se incorporarán las coordenadas representativas
obtenidas a partir de las geometrías municipales de INEGI. Estas
coordenadas permitirán realizar las consultas históricas a Open-Meteo.

## 2. Integración geográfica para Open-Meteo

Open-Meteo realiza las consultas climáticas a partir de coordenadas
geográficas. Por esta razón, cada municipio con actividad agrícola debe
estar asociado con una latitud y longitud.

Las coordenadas fueron obtenidas previamente a partir de las geometrías
municipales de INEGI mediante un punto representativo dentro de cada
municipio.

En esta etapa se integran:

- Los municipios con actividad agrícola.
- Las coordenadas municipales obtenidas de INEGI.

La integración utiliza la llave:

`(Idestado, Idmunicipio)`

El resultado constituye el conjunto de municipios que será enviado a
Open-Meteo para consultar el clima histórico de 2003 a 2025.

In [5]:
ruta_coordenadas = (
    INEGI_INTERIM_DIR
    / "municipios_coordenadas_inegi.csv"
)

municipios_coordenadas = pd.read_csv(
    ruta_coordenadas
)

municipios_coordenadas.head()

,Idestado,Idmunicipio,cvegeo,Nommunicipio,Latitud,Longitud
0,1,1,1001,Aguascalientes,21.848487,-102.305449
1,1,8,1008,San José de Gracia,22.154297,-102.524108
2,1,11,1011,San Francisco de los Romo,22.021698,-102.251799
3,1,9,1009,Tepezalá,22.243608,-102.192713
4,1,7,1007,Rincón de Romos,22.250711,-102.348371


In [6]:
print(
    f"Municipios con coordenadas INEGI: "
    f"{len(municipios_coordenadas):,}"
)

print(
    "Coordenadas faltantes:",
    municipios_coordenadas[
        ["Latitud", "Longitud"]
    ].isna().sum().sum()
)

print(
    "Claves duplicadas:",
    municipios_coordenadas.duplicated(
        ["Idestado", "Idmunicipio"]
    ).sum()
)

Municipios con coordenadas INEGI: 2,478
Coordenadas faltantes: 0
Claves duplicadas: 0


## 3. Selección de municipios para la consulta climática

No todos los municipios disponibles en la información geográfica de
INEGI forman parte del conjunto agrícola del proyecto.

Por ello, se conservan únicamente los municipios que presentan
actividad agrícola y se incorporan sus coordenadas geográficas.

Esta operación produce el conjunto definitivo de municipios que será
utilizado para realizar las consultas climáticas.

In [7]:
municipios_clima = preparar_municipios_clima(
    municipios_agricolas,
    municipios_coordenadas,
)

municipios_clima.head()

2026-10-02 17:03:31.963 | INFO     | proyecto_agronomia.features:preparar_municipios_clima:439 - Municipios agrícolas: 2,452
2026-10-02 17:03:31.964 | INFO     | proyecto_agronomia.features:preparar_municipios_clima:443 - Municipios sin coordenadas: 0


,Idestado,Idmunicipio,Nommunicipio,Latitud,Longitud
0,1,1,Aguascalientes,21.848487,-102.305449
1,1,5,Jesús María,21.928961,-102.461647
2,1,10,El Llano,21.928593,-102.000752
3,1,11,San Francisco de Los Romo,22.021698,-102.251799
4,1,3,Calvillo,21.911174,-102.713168


In [8]:
print(
    f"Municipios agrícolas: "
    f"{len(municipios_agricolas):,}"
)

print(
    f"Municipios preparados para Open-Meteo: "
    f"{len(municipios_clima):,}"
)

print(
    f"Entidades federativas: "
    f"{municipios_clima['Idestado'].nunique()}"
)

print(
    "Municipios sin latitud:",
    municipios_clima["Latitud"].isna().sum()
)

print(
    "Municipios sin longitud:",
    municipios_clima["Longitud"].isna().sum()
)

print(
    "Claves duplicadas:",
    municipios_clima.duplicated(
        ["Idestado", "Idmunicipio"]
    ).sum()
)

Municipios agrícolas: 2,452
Municipios preparados para Open-Meteo: 2,452
Entidades federativas: 32
Municipios sin latitud: 0
Municipios sin longitud: 0
Claves duplicadas: 0


### Resultado de la integración

Los municipios con actividad agrícola pudieron asociarse con sus
respectivas coordenadas geográficas.

El conjunto resultante contiene la información necesaria para realizar
las consultas a Open-Meteo:

- identificador de estado;
- identificador de municipio;
- nombre del municipio;
- latitud;
- longitud.

No se encontraron municipios agrícolas sin coordenadas, por lo que el
conjunto completo puede utilizarse en la etapa de adquisición climática.

A continuación se consultarán los datos históricos de Open-Meteo para
el periodo 2003–2025.

In [9]:
from proyecto_agronomia.dataset import (
    descargar_clima_todos_estados,
    obtener_clima_lote,
)

## 4. Consulta de datos climáticos con Open-Meteo

Una vez definidos los municipios y sus coordenadas, se utiliza
**Open-Meteo Historical Weather API** para obtener información
climática histórica.

La consulta utiliza la latitud y longitud representativa de cada
municipio y solicita las siguientes variables diarias:

- Temperatura media (`temperature_2m_mean`).
- Temperatura máxima (`temperature_2m_max`).
- Temperatura mínima (`temperature_2m_min`).
- Precipitación acumulada (`precipitation_sum`).
- Horas de precipitación (`precipitation_hours`).
- Evapotranspiración de referencia FAO (`et0_fao_evapotranspiration`).

Antes de ejecutar la descarga nacional se realiza una consulta pequeña
para verificar la estructura y contenido de la respuesta de la API.

In [10]:
municipios_prueba = municipios_clima.head(2).copy()

municipios_prueba

,Idestado,Idmunicipio,Nommunicipio,Latitud,Longitud
0,1,1,Aguascalientes,21.848487,-102.305449
1,1,5,Jesús María,21.928961,-102.461647


In [11]:
#datos_prueba = obtener_clima_lote(
#    municipios_prueba,
#    anio_inicio=2024,
#    anio_fin=2024,
#)

#print(
#    f"Respuestas recibidas: "
#    f"{len(datos_prueba)}"
#)

#print(
#    f"Tipo de respuesta: "
#    f"{type(datos_prueba).__name__}"
#)

## 4. Datos obtenidos de Open-Meteo

La adquisición de los datos climáticos se realiza mediante
**Open-Meteo Historical Weather API** utilizando las coordenadas
representativas de los municipios agrícolas.

Debido al volumen de información solicitado, la descarga completa se
realiza por entidad federativa y mediante lotes pequeños de municipios.

Los resultados se almacenan progresivamente en `data/raw/clima`,
permitiendo:

- conservar los datos obtenidos después de cada lote;
- evitar descargar nuevamente municipios completos;
- reanudar una descarga interrumpida;
- reducir solicitudes innecesarias a la API.

A continuación se inspecciona uno de los archivos climáticos obtenidos
durante este proceso.

In [12]:
from proyecto_agronomia.config import (
    ANIO_INICIAL,
    ANIO_FINAL,
    CLIMA_RAW_DIR,
    INEGI_INTERIM_DIR,
)

In [13]:
ruta_clima_ejemplo = (
    CLIMA_RAW_DIR
    / "clima_01_aguascalientes.csv"
)

clima_ejemplo = pd.read_csv(
    ruta_clima_ejemplo
)

clima_ejemplo.head()

,Anio,Temperatura_media,Temperatura_maxima,Temperatura_minima,Precipitacion_anual,Dias_con_lluvia,Horas_precipitacion,Evapotranspiracion_anual,Idestado,Idmunicipio,Nommunicipio,Latitud,Longitud
0,2003,19.62,36.0,1.5,664.7,112,820.0,1848.37,1,1,Aguascalientes,21.848487,-102.305449
1,2004,18.52,34.2,0.3,651.8,141,889.0,1756.49,1,1,Aguascalientes,21.848487,-102.305449
2,2005,19.60,36.5,3.4,379.7,107,571.0,1910.89,1,1,Aguascalientes,21.848487,-102.305449
3,2006,19.35,34.4,-1.3,584.1,126,798.0,1841.46,1,1,Aguascalientes,21.848487,-102.305449
4,2007,19.43,34.5,2.9,575.0,116,754.0,1847.47,1,1,Aguascalientes,21.848487,-102.305449


In [14]:
print(
    f"Observaciones: "
    f"{len(clima_ejemplo):,}"
)

print(
    f"Municipios: "
    f"{clima_ejemplo['Idmunicipio'].nunique():,}"
)

print(
    f"Periodo: "
    f"{clima_ejemplo['Anio'].min()}-"
    f"{clima_ejemplo['Anio'].max()}"
)

print(
    "Claves duplicadas:",
    clima_ejemplo.duplicated(
        ["Anio", "Idestado", "Idmunicipio"]
    ).sum()
)

print(
    "Valores faltantes:",
    clima_ejemplo.isna().sum().sum()
)

Observaciones: 253
Municipios: 11
Periodo: 2003-2025
Claves duplicadas: 0
Valores faltantes: 0


### Resultado

El archivo climático de Aguascalientes contiene información completa
para 11 municipios agrícolas durante los 23 años del periodo de estudio
(2003–2025), dando un total de 253 observaciones anuales.

Cada observación representa la información climática de un municipio
durante un año y se identifica mediante la llave:

`(Anio, Idestado, Idmunicipio)`

Esta estructura es compatible con la información agrícola y permitirá
posteriormente integrar ambas fuentes de datos.

In [18]:
resumen_descarga = descargar_clima_todos_estados(
    municipios_clima,
    tamano_lote=2,
    anio_inicio=ANIO_INICIAL,
    anio_fin=ANIO_FINAL,
    pausa=30,
)

resumen_descarga


2026-10-02 18:39:59.627 | INFO     | proyecto_agronomia.dataset:descargar_clima_todos_estados:569 - Descarga climática nacional | Periodo: 2003-2025 | Estados: 32 | Municipios agrícolas: 2,452
2026-10-02 18:39:59.630 | INFO     | proyecto_agronomia.dataset:descargar_clima_todos_estados:587 - Procesando estado 01/32: AGUASCALIENTES | 11 municipios
2026-10-02 18:39:59.633 | INFO     | proyecto_agronomia.dataset:descargar_clima_estado:301 - Estado: aguascalientes | Municipios agrícolas: 11
2026-10-02 18:39:59.641 | INFO     | proyecto_agronomia.dataset:descargar_clima_estado:329 - Municipios ya descargados: 11
2026-10-02 18:39:59.644 | INFO     | proyecto_agronomia.dataset:descargar_clima_estado:357 - Municipios pendientes: 0
2026-10-02 18:39:59.644 | SUCCESS  | proyecto_agronomia.dataset:descargar_clima_estado:362 - aguascalientes: estado completo. No se requieren solicitudes.
2026-10-02 18:39:59.647 | SUCCESS  | proyecto_agronomia.dataset:descargar_clima_todos_estados:639 - AGUASCALIENT

RuntimeError: Descarga detenida por límite de solicitudes en chiapas. Reanudar más tarde.